#Data Analyst Assignment

## Introduction

You are working with a US retail customer on a pilot deployment.  They are using technology to track their merchandise throughout their supply chain.  The flow of their supply is:

*   **DC 1:**  Orders are filled and palletized.
*   **Truck:** Pallets travel from the DC 1 to DC 2 via semi-truck.
*   **DC 2:**  Pallets are unloaded, and additional merchandise may be added.  They are then reloaded onto a new truck.
*   **Truck:** Pallets travel from DC 2 to the Store.
*   **Store:** Pallets are unloaded, cases are removed, and stocked, and the empty cases are left behind the building awaiting pickup.

Your job is to dig into the data and find compelling insights to show the value fo the technology and help move the contract from a pilot into a full scaled deployment.



---

## Part 0: Imports

Import necessary packages and

In [ ]:
#Alejandro Cordero Bolaños
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# YOUR CODE HERE:
import pandas as pd
import numpy as np
import datetime as dt
import plotly.express as px
import plotly.graph_objects as go
df=pd.read_excel("/content/drive/MyDrive/Elem1/Assignment_1.xlsx")
df.info()

### Dataset Overview

* Site:  A large space that could contain multiple readers. Ex: DC 1.
* Zone:  Point of interest. These represent areas in which repeaters are installed. These can be thought of as sub-zones.  Ex: Dock Doors.
* Asset ID: The unique ID of the asset.
* Asset Type: The type of thing that is detected (ie tote).
* Device ID: The unique gateway reader ID that detected the device in the zone (there can be multiple in one zone).
* Time est: The time in EST.
* Lon: Longituge
* Lat: Latitude
* Temperature_C / F: Temperature in Celsius, Fahrentheit

## PART 1: Data Overview

### Question 1:

* How many unique cases were we tracking throughout this pilot. (1 pt)
* What are the unique zones we could see (1 pt)
* How many POI's are in each Zone. (2 pts)


In [ ]:
# YOUR CODE HERE:
print("Casos únicos:", df['asset_id'].nunique())
print("Zonas únicas:", df['Zone'].unique()) #Al principio se indica que estos son los POIs

### Question 2:

* What is the temperature range we see?  (1pt)
* Where is temperature the highest and lowest (1pt)

In [ ]:
# YOUR CODE HERE:
print("Rango de temperatura:",df['Temperature_C'].max() - df['Temperature_C'].min())
print("Temperatura más alta:",df['Temperature_C'].max())
print("Temperatura más baja:",df['Temperature_C'].min())

## Part 2: The Journey of a Case

### Question 3:

* Create a visualization that shows where a case was at over time at the zone or POI level. Imagine that this would be included in your presentation to the customer. (Non techical audience) (3 pts)

In [ ]:
# YOUR CODE HERE:
fig = px.line(
    df,
    x='time_est',
    y='Zone',
    color='asset_id',
    title='Trayectoria por zonas a través del tiempo'
)
fig.show()

### Question 4:

* Visualize how the temperatue changes over time along its journey.  Imagine that this would be included in your presentation to the customer. (Non techical audience) (4 pts)



In [ ]:
# YOUR CODE HERE:
fig2 = px.line(
    df,
    x='time_est',
    y='Temperature_C',
    color='asset_id',
    title='Temperatura a lo largo del tiempo'
)
fig2.show()

### Question 5:
* Visualize the lon lat data on a map to show how the case traveled.  You may incorporate any other additional information to make this more impactful. Imagine that this would be included in your presentation to the customer. (Non techical audience) (5 pts)

**Do not worry if this looks like non-sense on a map.  Ex:  The trip may appear to occur over water or in a forest because this is a toy dataset.**

In [ ]:
# YOUR CODE HERE:
fig3 = px.scatter_map(
    df,
    lat='lat',
    lon='lng',
    color='asset_id',
    hover_name='Zone',
    zoom=11,
    height=600
)
fig3.show()

# Part 3: Customer Questions


### Question 6:

The customer wants to understand the efficieny of ther DC operations.
* Based on what you see in the data, (all zones except for STORE), which parts of their operation are most & least "efficient? (10 pts)

In [ ]:
# YOUR CODE HERE
df2 = df[df['Site'] != 'Store'].copy()
df2 = df2.sort_values(['asset_id', 'time_est'])
df2['difftiempozona'] = df2.groupby(['asset_id', 'Zone'])['time_est'].diff()
df2['diffsegzona'] = df2['difftiempozona'].dt.total_seconds()
eficienciazona = df2.groupby('Zone')['diffsegzona'].mean().sort_values(ascending=False)
print("Promedio de tiempo en cada zona (seg): \n", eficienciazona)

YOUR TEXT ANSWER HERE

### Question 7:

The customer wants to understand the stocking efficiency in stores.
* Based on what you see in the data, how quickly did the store unload and stock the merchandise. (5 pts)
* How could this be converted in a KPI that a regional manager could track?  (5 pts)

In [ ]:
# YOUR CODE HERE
df3 = df[df['Site'] == 'Store'].copy()

recibido = df3[df3['Zone'] == 'receiving_Store'].groupby('asset_id')['time_est'].min().rename('horarecibido')

zonas = ['store_back_Store', 'store_front_Store']
df4 = df3.merge(recibido, on='asset_id')
df5 = df4[df4['time_est'] >= df4['horarecibido']]

horastock = df5[df5['Zone'].isin(zonas)].groupby('asset_id')['time_est'].min().rename('stock')

rendimiento = pd.concat([recibido, horastock], axis=1).dropna()
rendimiento['tiempototal'] = (rendimiento['stock'] - rendimiento['horarecibido']).dt.total_seconds() / 60.0

print("Velocidad promedio de abastecimiento (min):", rendimiento['tiempototal'].mean())

YOUR TEXT ANSWER HERE

### Question 8:

Please explain what you would ask for and what you will do with this data, given that you can talk with the following people (no code needed):


YOUR TEXT ANSWER HERE
* a. Datos de inventario
* b. Datos a tiempo real

## Part 4: Bonus Insights

### Question 8

The customer is open to hearing about additional insights you found in the data above and beyond what they asked for.
* Based on what you can see in the data, are there any other interesting insights that the customer may want to hear about? (Up to 15 bonus points)



In [ ]:
# YOUR CODE HERE
# Algunas métricas interesantes a las que se prestan los datos pueden ser:
# - Horas pico o con mayor actividad
df['hora'] = df['time_est'].dt.hour
horaspico = df['hora'].value_counts().sort_index()
print("Registros por hora (Actividad):", horaspico)
# - Desbalance en el uso de montacargas
# - Trayectorias inusuales

YOUR TEXT ANSWER HERE
